In [1]:
!pip install -q langchain_community langchain_text_splitters langchain_openai langchain_core
!pip install -U pip setuptools wheel
!pip install -U langchain langchain-community langchain-openai langchain-text-splitters langchain-classic pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 123.5 MB/s  0:00:00
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.2
    Uninstalling langgraph-checkpoint-4.0.2:
      Successfully uninstalled langgraph-checkpoint-4.0.2
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.9
    Uninstalling langgraph-1.1.9:
      Successfully uninstalled langgraph-1.1.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.15
    Uninstalling langchain-1.2.15:
      Successfully uninstalled langchain-1.2.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain]


In [2]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [3]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [5]:
def build_rag_chain(file_path: str):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found.")

    # 1) Ingestion
    docs = TextLoader(file_path).load()

    # 2) Splitting
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50
    )
    chunks = splitter.split_documents(docs)

    # 3) Embedding + Indexing
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(chunks, embeddings)

    # 4) Retrieval
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 5) Generation
    prompt = ChatPromptTemplate.from_template(
        """Answer the question using only the context below.
If the answer is not in the context, say: "I do not know based on the provided document."

Context:
{context}

Question:
{question}
"""
    )

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain, retriever

In [7]:
def main():
    rag_chain, retriever = build_rag_chain("it_helpdesk_kb.txt")

    #question = "How do I reset my password?"
    question = "When does VPN lock?"
    #question = "What causes account lockouts?"
    #question = "Who approves software installation?"
    #question = "What should I do if I lose my device?"
    #question = "How are tickets prioritized?"
    #question = "When should a ticket be escalated?"
    #question = "When is 2+2?"
    answer = rag_chain.invoke(question)
    source_docs = retriever.invoke(question)

    print("QUESTION:")
    print(question)
    print("\nANSWER:")
    print(answer)
    print("\nRETRIEVED CHUNKS:")
    for i, doc in enumerate(source_docs, start=1):
        print(f"\nChunk {i}:")
        print(doc.page_content[:300])

In [8]:
main()

QUESTION:
When does VPN lock?

ANSWER:
VPN access may lock after 5 failed login attempts.

RETRIEVED CHUNKS:

Chunk 1:
Question: When does VPN access lock?
Answer: VPN access may lock after 5 failed login attempts.
Question: Who approves software installs?

Chunk 2:
may disconnect after periods of inactivity or unstable home internet connection. If VPN access locks after 5 failed login attempts, the employee should wait the specified lockout period or contact the Helpdesk if the issue continues.

Chunk 3:
User accounts may lock after multiple failed sign-in attempts. For standard network accounts, lockout typically occurs after 5 failed attempts within a short time window. VPN access may also lock after 5 failed login attempts. In most cases, locked accounts automatically unlock after 15 minutes
